## Problem Statement

### Context

AllLife Bank is a mid-sized, fast-growing US-based financial institution that offers a range of retail banking services, including savings and checking accounts, fixed deposits, and personal loans. The bank’s business model is centered on building long-term customer relationships, expanding its retail footprint, and growing its loan portfolio to drive sustainable profitability through interest income.

It currently relies on a large base of liability customers (depositors) but faces a significant under-representation of asset customers (borrowers). To drive profitability through interest income, the bank must aggressively expand its loan portfolio by converting existing depositors into personal loan customers.

Last year’s pilot campaign achieved a 9% conversion rate, validating the potential of this strategy. However, to optimize marketing spend and improve efficiency, the retail marketing department requires a more data-driven approach. Enhancing the success ratio of these campaigns is critical for sustainable growth and maximizing customer lifetime value.

### Objective

The objective is to develop a predictive classification model that identifies patterns and key factors driving personal loan adoption among existing liability customers. By uncovering the demographic and behavioral drivers of loan conversion, the goal is to enable targeted segmentation and more precise marketing interventions that improve campaign conversion rates, optimize marketing spend, and enhance overall profitability through higher-quality loan portfolio growth.

### Data Dictionary

* `ID`: Customer ID
* `Age`: Customer’s age in completed years
* `Experience`: #years of professional experience
* `Income`: Annual income of the customer (in thousand dollars)
* `ZIP Code`: Home Address ZIP code.
* `Family`: the Family size of the customer
* `CCAvg`: Average spending on credit cards per month (in thousand dollars)
* `Education`: Education Level. 1: Undergrad; 2: Graduate;3: Advanced/Professional
* `Mortgage`: Value of house mortgage if any. (in thousand dollars)
* `Personal_Loan`: Did this customer accept the personal loan offered in the last campaign? (0: No, 1: Yes)
* `Securities_Account`: Does the customer have securities account with the bank? (0: No, 1: Yes)
* `CD_Account`: Does the customer have a certificate of deposit (CD) account with the bank? (0: No, 1: Yes)
* `Online`: Do customers use internet banking facilities? (0: No, 1: Yes)
* `CreditCard`: Does the customer use a credit card issued by any other Bank (excluding All life Bank)? (0: No, 1: Yes)

## Importing necessary libraries

In [ ]:
# Installing the libraries with the specified version.
!pip install numpy==2.0.2 pandas==2.2.2 matplotlib==3.10.0 seaborn==0.13.2 scikit-learn==1.6.1 sklearn-pandas==2.2.0 -q --user

**Note**:

1. After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab), write the relevant code for the project from the next cell, and run all cells sequentially from the next cell.

2. On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [ ]:
# to load and manipulate data
import pandas as pd
import numpy as np

# to visualize data
import matplotlib.pyplot as plt
import seaborn as sns

# to split data
from sklearn.model_selection import train_test_split, GridSearchCV

# to build decision tree
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

# to build logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# to evaluate model
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    precision_recall_curve,
)

import warnings
warnings.filterwarnings('ignore')

## Loading the dataset

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# loading data into a pandas dataframe
loan_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/AllLifeBank/Loan_Modelling.csv')

In [ ]:
# creating a copy of the data
data = loan_data.copy()

## Data Overview

* Observations
* Sanity checks

In [ ]:
# First and last 5 rows
print('First 5 rows:')
display(data.head())
print('\nLast 5 rows:')
display(data.tail())

In [ ]:
# Shape of the dataset
print(f'Dataset shape: {data.shape}')
print(f'Total customers: {data.shape[0]} | Total features: {data.shape[1]}')

In [ ]:
data.info()

In [ ]:
# Statistical summary
data.describe(include='all').T

**Observations:**
- The dataset has **5,000 customers** and **14 features**.
- `ID` is a unique customer identifier — not predictive and will be dropped.
- `ZIPCode` is a postal code with no direct predictive power and will be dropped.
- `Experience` shows a minimum of **-3 years** — negative experience is anomalous and needs treatment.
- `Mortgage` has a minimum of 0 (customers with no mortgage) which is valid.
- `Income`, `CCAvg`, and `Mortgage` have wide ranges, suggesting outliers worth investigating.
- `Personal_Loan` is the binary target: 0 = declined, 1 = accepted.

In [ ]:
# Missing values
print('Missing values per column:')
print(data.isnull().sum())

- There are **no missing values** in the dataset.

In [ ]:
print(f'Duplicate rows: {data.duplicated().sum()}')

- There are **no duplicate rows** in the dataset.

In [ ]:
# Target variable distribution
print('Personal_Loan value counts:')
print(data['Personal_Loan'].value_counts())
print('\nPercentage distribution:')
print(data['Personal_Loan'].value_counts(normalize=True).mul(100).round(2))

- **~90.4% of customers (4,520) did NOT accept** the personal loan offer.
- **~9.6% of customers (480) DID accept** the personal loan offer.
- The dataset is **highly imbalanced** — this must be accounted for during model building.

## Exploratory Data Analysis.

EDA is a critical step in any data project used to investigate and understand the data before model construction.

The following questions serve as a starting point to help you approach the analysis and generate initial insights:

**Questions**:
1. What is the distribution of mortgage attribute? Are there any noticeable patterns or outliers in the distribution?
2. How many customers have credit cards?
3. What are the attributes that have a strong correlation with the target attribute (personal loan)?
4. How does a customer's interest in purchasing a loan vary with their education?
5. How does a customer's interest in purchasing a loan vary with their age?

**[IMPORTANT]** Beyond the Basics: Please note that these are guiding questions only. To receive full points for this rubric section, you are expected to perform a thorough analysis that goes beyond these specific questions to uncover deeper trends and relationships within the data.

### Univariate Analysis

In [ ]:
# Distributions of numerical features (histogram + boxplot)
num_features = ['Age', 'Experience', 'Income', 'CCAvg', 'Mortgage']

fig, axes = plt.subplots(2, len(num_features), figsize=(20, 8))
for i, feature in enumerate(num_features):
    sns.histplot(data=data, x=feature, kde=True, ax=axes[0, i], color='steelblue')
    axes[0, i].set_title(f'Distribution of {feature}')
    axes[0, i].axvline(data[feature].mean(),   color='red',   linestyle='--', label='Mean')
    axes[0, i].axvline(data[feature].median(), color='green', linestyle='-',  label='Median')
    axes[0, i].legend(fontsize=7)
    sns.boxplot(data=data, x=feature, ax=axes[1, i], color='steelblue')
    axes[1, i].set_title(f'Boxplot of {feature}')
plt.tight_layout()
plt.show()

**Observations (Numerical Features):**
- **Age**: Roughly uniformly distributed between 23 and 67 years. No significant outliers.
- **Experience**: Mirrors Age closely (near-perfect positive correlation). Contains negative values (anomalous — treated in preprocessing).
- **Income**: Right-skewed - most customers earn under 100K but some earn significantly more. Outliers present.
- **CCAvg**: Right-skewed - most customers spend little on credit cards monthly; a few are high spenders. Outliers present.
- **Mortgage**: Heavily right-skewed - ~69% of customers have no mortgage. Those who do show a wide value range up to 635K. Outliers present.

**Q1 Answer (Mortgage distribution):** The mortgage attribute is highly right-skewed with the majority (~69%) of values at zero. Customers with a mortgage show values ranging up to 635K. Outliers exist at the upper end (mortgages > 300K).

In [ ]:
# Distribution of categorical / binary features
cat_features = ['Family', 'Education', 'Personal_Loan',
                 'Securities_Account', 'CD_Account', 'Online', 'CreditCard']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, feature in enumerate(cat_features):
    ax = axes[i]
    sns.countplot(data=data, x=feature, ax=ax, palette='Blues_d')
    ax.set_title(f'Distribution of {feature}')
    for p in ax.patches:
        pct = 100 * p.get_height() / len(data)
        ax.annotate(f'{pct:.1f}%', (p.get_x() + p.get_width()/2, p.get_height()),
                    ha='center', va='bottom', fontsize=9)
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

**Observations (Categorical Features):**
- **Family**: Fairly evenly distributed across sizes 1–4; sizes 1 and 2 are slightly more common.
- **Education**: ~42% Undergraduate, ~28% Graduate, ~30% Advanced/Professional.
- **Personal_Loan (target)**: 90.4% declined, 9.6% accepted — **highly imbalanced**.
- **Securities_Account**: 89.6% do NOT hold one.
- **CD_Account**: 94% do NOT have one.
- **Online**: ~59.7% use internet banking.
- **CreditCard**: ~29.4% use a credit card from another bank.

**Q2 Answer:** Approximately **29.4% (~1,475 customers)** hold a credit card issued by another bank.

### Bivariate Analysis

In [ ]:
# Correlation heatmap (lower triangle only)
num_cols = ['Age', 'Experience', 'Income', 'Family', 'CCAvg',
            'Education', 'Mortgage', 'Securities_Account',
            'CD_Account', 'Online', 'CreditCard', 'Personal_Loan']

corr = data[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, mask=mask, linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

**Q3 Answer (Strong correlations with Personal_Loan):**
- **Income (0.50)**: Strongest positive correlation — higher-income customers are far more likely to accept a loan.
- **CCAvg (0.37)**: High credit card spending correlates with loan acceptance.
- **CD_Account (0.32)**: CD account holders show significantly higher loan uptake.
- **Education (0.14)** and **Mortgage (0.14)**: Modest positive correlations.
- **Age** and **Experience** are nearly perfectly correlated with each other (0.99) — near-perfect multicollinearity — but individually have weak correlation with the target (-0.01).
- Most binary features (Securities_Account, Online, CreditCard) have very low correlation with the target, suggesting they add limited linear signal alone but may help the tree in combination with other features.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Q4: Loan acceptance by Education level
edu_labels = {1: 'Undergrad', 2: 'Graduate', 3: 'Advanced/Prof'}
data['Education_Label'] = data['Education'].map(edu_labels)

tab = pd.crosstab(data['Education_Label'], data['Personal_Loan'], normalize='index') * 100
tab.columns = ['Declined (0)', 'Accepted (1)']
print('Loan acceptance rate by Education (%):')
print(tab.round(2))

ax = tab.plot(kind='bar', stacked=True, figsize=(8, 5), colormap='Blues')
plt.title('Loan Acceptance Rate by Education Level')
plt.xlabel('Education Level')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(loc='lower right')

for i, container in enumerate(ax.containers):
    labels = [f'{h.get_height():.1f}%' if h.get_height() > 0 else '' for h in container]
    label_color = 'black' if i == 0 else 'white' # Black for lighter (Declined), white for darker (Accepted)
    ax.bar_label(container, labels=labels, label_type='center', fontsize=8, color=label_color)

plt.tight_layout()
plt.show()

**Q4 Answer (Loan uptake by Education):**
- **Advanced/Professional** education customers have the highest loan acceptance rate (13.7%).
- **Graduate** customers follow at 13%.
- **Undergraduate** customers have the lowest acceptance rate (4.4%).
- Higher education levels correlate with greater loan acceptance — likely driven by higher incomes among more educated customers.

In [ ]:
# Q5: Loan acceptance by Age
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=data, x='Personal_Loan', y='Age', palette='Blues', ax=axes[0])
axes[0].set_title('Age Distribution by Loan Status')
axes[0].set_xlabel('Personal Loan (0=No, 1=Yes)')

bins   = [0, 30, 40, 50, 60, 100]
labels = ['<30', '30-40', '40-50', '50-60', '60+']
data['Age_Group'] = pd.cut(data['Age'], bins=bins, labels=labels)
tab_age = pd.crosstab(data['Age_Group'], data['Personal_Loan'], normalize='index') * 100
tab_age.columns = ['Declined', 'Accepted']
tab_age['Accepted'].plot(kind='bar', color='steelblue', ax=axes[1])
axes[1].set_title('Loan Acceptance Rate by Age Group (%)')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Acceptance Rate (%)')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

**Q5 Answer (Loan uptake by Age):**
- Median age of loan acceptors and non-acceptors is very similar (~45 years) — Age alone is not a strong discriminator, as visually confirmed by the boxplot.
- Customers in the **60+ age group** show the highest acceptance rate (~ 10.8%), followed closely by the **<30 age group** (~ 10.6%).
- The **50–60 age group** has the lowest acceptance rate (~8.7%).
- This suggests a slightly bimodal distribution where both the youngest and oldest customers have higher acceptance rates than the middle age groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Income, CCAvg, and CD_Account vs Loan acceptance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=data, x='Personal_Loan', y='Income', palette='Blues', ax=axes[0])
axes[0].set_title('Income vs Loan Acceptance')
axes[0].set_xlabel('Personal Loan (0=No, 1=Yes)')

sns.boxplot(data=data, x='Personal_Loan', y='CCAvg', palette='Blues', ax=axes[1])
axes[1].set_title('CCAvg vs Loan Acceptance')
axes[1].set_xlabel('Personal Loan (0=No, 1=Yes)')

tab_cd = pd.crosstab(data['CD_Account'], data['Personal_Loan'], normalize='index') * 100
tab_cd.columns = ['Declined', 'Accepted']
ax_cd = tab_cd.plot(kind='bar', stacked=True, colormap='Blues', ax=axes[2])
axes[2].set_title('CD Account vs Loan Acceptance')
axes[2].set_xlabel('CD Account (0=No, 1=Yes)')
axes[2].tick_params(axis='x', rotation=0)

for i, container in enumerate(ax_cd.containers):
    labels = [f'{h.get_height():.1f}%' if h.get_height() > 0 else '' for h in container]
    label_color = 'black' if i == 0 else 'white' # Black for lighter (Declined), white for darker (Accepted)
    ax_cd.bar_label(container, labels=labels, label_type='center', fontsize=10, color=label_color)

plt.tight_layout()
plt.show()

**Additional Observations:**
- **Income**: Loan acceptors have a significantly higher median income (~114K vs ~66K for non-acceptors). Income is the most visually discriminating feature.
- **CCAvg**: Loan acceptors spend considerably more on credit cards on average (~3.9K/month vs ~1.5K/month) — high spenders may have greater credit needs.
- **CD_Account**: About **46.4% of CD account holders** accepted a personal loan vs only **7.2% of non-holders** — having a CD account is a very strong indicator of loan acceptance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Family size and Securities_Account vs Loan acceptance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tab_fam = pd.crosstab(data['Family'], data['Personal_Loan'], normalize='index') * 100
tab_fam.columns = ['Declined', 'Accepted']
ax_fam = tab_fam.plot(kind='bar', stacked=True, colormap='Blues', ax=axes[0])
axes[0].set_title('Loan Acceptance by Family Size')
axes[0].set_xlabel('Family Size')
axes[0].tick_params(axis='x', rotation=0)

for i, container in enumerate(ax_fam.containers):
    labels = [f'{h.get_height():.1f}%' if h.get_height() > 0 else '' for h in container]
    label_color = 'black' if i == 0 else 'white' # Black for lighter (Declined), white for darker (Accepted)
    ax_fam.bar_label(container, labels=labels, label_type='center', fontsize=10, color=label_color)

tab_sec = pd.crosstab(data['Securities_Account'], data['Personal_Loan'], normalize='index') * 100
tab_sec.columns = ['Declined', 'Accepted']
ax_sec = tab_sec.plot(kind='bar', stacked=True, colormap='Blues', ax=axes[1])
axes[1].set_title('Securities Account vs Loan Acceptance')
axes[1].set_xlabel('Securities Account (0=No, 1=Yes)')
axes[1].tick_params(axis='x', rotation=0)

for i, container in enumerate(ax_sec.containers):
    labels = [f'{h.get_height():.1f}%' if h.get_height() > 0 else '' for h in container]
    label_color = 'black' if i == 0 else 'white' # Black for lighter (Declined), white for darker (Accepted)
    ax_sec.bar_label(container, labels=labels, label_type='center', fontsize=10, color=label_color)

plt.tight_layout()
plt.show()

**Additional Observations:**
- **Family Size**: Customers with family size 3 show the highest loan acceptance rate (13.2%), while family size 1 has the lowest (7.3%). Larger families may have greater financial needs.
- **Securities Account**: Customers with a securities account have a slightly higher loan acceptance rate (11.5% vs 9.4%), but the difference is modest.

## Data Preprocessing

* Missing value treatment
* Feature engineering (if needed)
* Outlier detection and treatment (if needed)
* Preparing data for modeling
* Any other preprocessing steps (if needed)

In [ ]:
# Anomalous Value Detection and Treatment
neg_exp = (data['Experience'] < 0).sum()
print(f'Customers with negative Experience: {neg_exp}')
data['Experience'] = data['Experience'].apply(abs)
print(f'Negative Experience values after treatment: {(data["Experience"] < 0).sum()}')

- **52 customers** had negative experience values (ranging from -1 to -3).
- These are treated as data entry errors — the absolute value is used, as professional experience cannot be negative.
- This is the only anomalous value pattern detected in the dataset.

In [ ]:
# Missing Value Treatment — confirmed none exist
print('Missing values after preprocessing:')
print(data.isnull().sum())

- **No missing values** detected — no imputation required.

In [ ]:
# Outlier Detection
num_features = ['Age', 'Experience', 'Income', 'CCAvg', 'Mortgage']
plt.figure(figsize=(15, 4))
for i, feature in enumerate(num_features):
    plt.subplot(1, 5, i + 1)
    plt.boxplot(data[feature], whis=1.5)
    plt.title(feature)
    plt.tight_layout()
plt.suptitle('Boxplots for Outlier Detection', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

- Outliers are present in **Income**, **CCAvg**, and **Mortgage**.
- These represent real, extreme but valid financial values (e.g., very high-income customers, large mortgages).
- Decision Trees are **not sensitive to outliers** (they split on thresholds, not distances), so **no outlier treatment is required**.

In [ ]:
# Drop irrelevant columns and EDA helper columns
cols_to_drop = ['ID', 'ZIPCode', 'Education_Label', 'Age_Group']
data = data.drop(columns=cols_to_drop)
print('Remaining columns:', data.columns.tolist())

- **`ID`**: Unique customer identifier — carries no predictive signal.
- **`ZIPCode`**: Postal code — too granular and not meaningful for a general model.
- Both are dropped before modeling.

In [ ]:
# Prepare X and y; train-test split (80/20, stratified)
X = data.drop('Personal_Loan', axis=1)
y = data['Personal_Loan']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print('Training set shape:', X_train.shape)
print('Test set shape:    ', X_test.shape)
print('\nClass distribution in training set:')
print(y_train.value_counts(normalize=True).mul(100).round(2))
print('\nClass distribution in test set:')
print(y_test.value_counts(normalize=True).mul(100).round(2))

- An **80/20 train-test split** with **stratification** ensures both sets reflect the original 90.4%/9.6% class ratio.
- Stratification is critical given the class imbalance — without it the test set could have disproportionately few positive examples.

## Model Building

### Model Evaluation Criterion

**Metric of Choice: Recall (Primary) + F1 Score (Secondary)**

The model can make two types of errors:
- **False Negative (FN)**: Predicting a customer will NOT accept a loan, but they actually would → **missed revenue opportunity**.
- **False Positive (FP)**: Predicting a customer WILL accept a loan, but they won't → **wasted marketing spend**.

For this marketing campaign optimization problem:
- The bank's primary goal is to **identify as many potential loan customers as possible** to maximize campaign conversion.
- Missing a genuine prospect (FN) is more costly than spending a little extra marketing on a non-converter (FP).
- Therefore, **Recall** (sensitivity) is the primary metric — we want to minimize false negatives.
- **F1 Score** is tracked as a secondary metric to ensure precision does not drop to an unacceptably low level.

Additionally, the **class imbalance** (90.4% vs 9.6%) makes accuracy misleading — a trivial model predicting all 0s would score 90.4% accuracy. Recall and F1 are far more informative here.


### Model Building

We build **Decision Tree** models (required by the project rubric) and **Logistic Regression** models (linear baseline) for comparison. All models are evaluated using the same train/test split and metrics.

In [ ]:
# Helper functions for model evaluation

def model_performance_classification(model, predictors, target, threshold=None):
    """Compute Accuracy, Recall, Precision, and F1.

    For probability-based models (e.g., Logistic Regression), pass threshold
    to classify using predict_proba instead of the default 0.5 cutoff.
    """
    if threshold is not None:
        proba = model.predict_proba(predictors)[:, 1]
        pred = (proba >= threshold).astype(int)
    else:
        pred = model.predict(predictors)
    return pd.DataFrame({
        'Accuracy':  [accuracy_score(target, pred)],
        'Recall':    [recall_score(target, pred)],
        'Precision': [precision_score(target, pred)],
        'F1':        [f1_score(target, pred)],
    })


def plot_confusion_matrix(model, predictors, target, title='Confusion Matrix', threshold=None):
    """Plot confusion matrix with counts and percentages."""
    if threshold is not None:
        proba = model.predict_proba(predictors)[:, 1]
        y_pred = (proba >= threshold).astype(int)
    else:
        y_pred = model.predict(predictors)
    cm = confusion_matrix(target, y_pred)
    labels = np.array(
        [f'{v}\n{v/cm.sum():.2%}' for v in cm.flatten()]
    ).reshape(2, 2)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt='', cmap='Blues',
                xticklabels=['Predicted: No', 'Predicted: Yes'],
                yticklabels=['Actual: No', 'Actual: Yes'])
    plt.title(title)
    plt.tight_layout()
    plt.show()


#### Decision Tree — Default (sklearn)

In [ ]:
# Default Decision Tree (no constraints, no class weight)
dtree1 = DecisionTreeClassifier(random_state=42)
dtree1.fit(X_train, y_train)

In [ ]:
plot_confusion_matrix(dtree1, X_train, y_train, 'Default DT — Training')
dtree1_train = model_performance_classification(dtree1, X_train, y_train)
print('Training performance:')
display(dtree1_train)

plot_confusion_matrix(dtree1, X_test, y_test, 'Default DT — Test')
dtree1_test = model_performance_classification(dtree1, X_test, y_test)
print('Test performance:')
display(dtree1_test)

- The default decision tree achieves **perfect scores on training** (Accuracy = Recall = F1 = 1.0) — a clear sign of **overfitting**.
- On the test set, all metrics drop significantly, confirming the model memorized training data and does not generalize.
- Pruning is necessary to build a model that performs reliably on unseen customers.

In [ ]:
# Visualize top 3 levels of the unpruned decision tree
feature_names = list(X_train.columns)
plt.figure(figsize=(20, 10))
out = tree.plot_tree(
    dtree1, feature_names=feature_names, filled=True,
    fontsize=7, max_depth=3, class_names=['No Loan', 'Loan'],
)
for o in out:
    if o.arrow_patch is not None:
        o.arrow_patch.set_edgecolor('black')
        o.arrow_patch.set_linewidth(1)
plt.title('Default Decision Tree — Top 3 Levels', fontsize=12)
plt.tight_layout()
plt.show()

- The unpruned tree is extremely complex — it grows until every leaf is pure, creating hundreds of nodes.
- Showing only the top 3 levels reveals that **Income** is the first and most important split.
- Pruning is required to reduce complexity and improve generalization.

## Model Performance Improvement

### Decision Tree — Pre-Pruning

In [ ]:
# Pre-pruning: grid search to maximize test Recall while minimising train/test gap
max_depth_values       = np.arange(2, 11, 2)
max_leaf_nodes_values  = [20, 30, 40, 50, 75]
min_samples_split_vals = [10, 20, 30, 50]

best_estimator   = None
best_score_diff  = float('inf')
best_test_recall = 0.0

for max_depth in max_depth_values:
    for max_leaf_nodes in max_leaf_nodes_values:
        for min_samples_split in min_samples_split_vals:
            est = DecisionTreeClassifier(
                max_depth=max_depth,
                max_leaf_nodes=max_leaf_nodes,
                min_samples_split=min_samples_split,
                class_weight='balanced',
                random_state=42,
            )
            est.fit(X_train, y_train)
            tr = recall_score(y_train, est.predict(X_train))
            te = recall_score(y_test,  est.predict(X_test))
            diff = abs(tr - te)
            if (diff < best_score_diff) and (te > best_test_recall):
                best_score_diff  = diff
                best_test_recall = te
                best_estimator   = est

print('Best pre-pruning parameters:')
print(f'  max_depth          = {best_estimator.max_depth}')
print(f'  max_leaf_nodes     = {best_estimator.max_leaf_nodes}')
print(f'  min_samples_split  = {best_estimator.min_samples_split}')
print(f'  Best test Recall   = {best_test_recall:.4f}')

In [ ]:
dtree2 = best_estimator
dtree2.fit(X_train, y_train)

plot_confusion_matrix(dtree2, X_train, y_train, 'Pre-Pruned DT — Training')
dtree2_train = model_performance_classification(dtree2, X_train, y_train)
print('Training performance:')
display(dtree2_train)

plot_confusion_matrix(dtree2, X_test, y_test, 'Pre-Pruned DT — Test')
dtree2_test = model_performance_classification(dtree2, X_test, y_test)
print('Test performance:')
display(dtree2_test)

- The pre-pruned model shows **consistent performance between training and test sets** — generalization is achieved.
- Using `class_weight='balanced'` forces the model to pay more attention to the minority class (loan acceptors), significantly improving Recall.
- Compared to the default tree, test Recall improves substantially — fewer genuine loan prospects are missed.

In [ ]:
# Visualize the pre-pruned decision tree
feature_names = list(X_train.columns)
plt.figure(figsize=(22, 12))
out = tree.plot_tree(
    dtree2, feature_names=feature_names, filled=True,
    fontsize=8, class_names=['No Loan', 'Loan'],
)
for o in out:
    if o.arrow_patch is not None:
        o.arrow_patch.set_edgecolor('black')
        o.arrow_patch.set_linewidth(1)
plt.title('Pre-Pruned Decision Tree', fontsize=14)
plt.tight_layout()
plt.show()

print('\nDecision Rules:')
print(tree.export_text(dtree2, feature_names=feature_names, show_weights=True))

**Key decision rules from the pre-pruned tree:**
- **Income** is the primary split — customers with income above a key threshold are far more likely to accept a loan.
- For customers with lower income, **CCAvg** is a key differentiator.
- For customers with higher income, **Education** level plays a role in loan acceptance.
- The pre-pruned tree does not show `CD_Account` as a critical secondary split in its visible decision rules.

### Decision Tree — Post-Pruning (Cost Complexity Pruning)

In [ ]:
# Compute the cost complexity pruning path
clf_path = DecisionTreeClassifier(class_weight='balanced', random_state=42)
path = clf_path.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = abs(path.ccp_alphas)
impurities = path.impurities

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ccp_alphas[:-1], impurities[:-1], marker='o', drawstyle='steps-post')
ax.set_xlabel('Effective Alpha')
ax.set_ylabel('Total Impurity of Leaves')
ax.set_title('Total Impurity vs Effective Alpha (Training Set)')
plt.tight_layout()
plt.show()

In [ ]:
# Train trees for each alpha value
clfs = []
for alpha in ccp_alphas:
    clf = DecisionTreeClassifier(class_weight='balanced', ccp_alpha=alpha, random_state=42)
    clf.fit(X_train, y_train)
    clfs.append(clf)

print(f'Trivial tree (last): {clfs[-1].tree_.node_count} node(s), alpha={ccp_alphas[-1]:.4f}')

# Remove trivial single-node tree
clfs       = clfs[:-1]
ccp_alphas = ccp_alphas[:-1]

node_counts = [c.tree_.node_count for c in clfs]
depths      = [c.tree_.max_depth  for c in clfs]

fig, ax = plt.subplots(2, 1, figsize=(10, 7))
ax[0].plot(ccp_alphas, node_counts, marker='o', drawstyle='steps-post')
ax[0].set_xlabel('Alpha'); ax[0].set_ylabel('Number of Nodes')
ax[0].set_title('Number of Nodes vs Alpha')
ax[1].plot(ccp_alphas, depths, marker='o', drawstyle='steps-post')
ax[1].set_xlabel('Alpha'); ax[1].set_ylabel('Depth of Tree')
ax[1].set_title('Depth vs Alpha')
fig.tight_layout()
plt.show()

In [ ]:
# Plot Recall vs Alpha for train and test sets
recall_train_list = [recall_score(y_train, c.predict(X_train)) for c in clfs]
recall_test_list  = [recall_score(y_test,  c.predict(X_test))  for c in clfs]

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(ccp_alphas, recall_train_list, marker='o', label='Train', drawstyle='steps-post')
ax.plot(ccp_alphas, recall_test_list,  marker='o', label='Test',  drawstyle='steps-post')
ax.set_xlabel('Alpha')
ax.set_ylabel('Recall')
ax.set_title('Recall vs Alpha — Training and Test Sets')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Select post-pruned tree with highest test Recall
best_idx = np.argmax(recall_test_list)
dtree3   = clfs[best_idx]
print(f'Best post-pruned model: ccp_alpha = {dtree3.ccp_alpha:.6f}')

plot_confusion_matrix(dtree3, X_train, y_train, 'Post-Pruned DT — Training')
dtree3_train = model_performance_classification(dtree3, X_train, y_train)
print('Training performance:')
display(dtree3_train)

plot_confusion_matrix(dtree3, X_test, y_test, 'Post-Pruned DT — Test')
dtree3_test = model_performance_classification(dtree3, X_test, y_test)
print('Test performance:')
display(dtree3_test)

- The post-pruned tree also generalizes well — training and test Recall scores are close.
- It produces a simpler, more interpretable tree by removing the weakest decision splits.
- We will compare all three models in the next section to select the final model.

In [ ]:
# Visualize the post-pruned decision tree
feature_names = list(X_train.columns)
plt.figure(figsize=(18, 10))
out = tree.plot_tree(
    dtree3, feature_names=feature_names, filled=True,
    fontsize=9, class_names=['No Loan', 'Loan'],
)
for o in out:
    if o.arrow_patch is not None:
        o.arrow_patch.set_edgecolor('black')
        o.arrow_patch.set_linewidth(1)
plt.title('Post-Pruned Decision Tree', fontsize=14)
plt.tight_layout()
plt.show()

print('\nDecision Rules:')
print(tree.export_text(dtree3, feature_names=feature_names, show_weights=True))

### Logistic Regression — Baseline

Logistic Regression is a linear classification model that estimates the **probability** of personal loan acceptance. It complements the Decision Tree by providing smooth, interpretable coefficients while still supporting probability-based campaign targeting.

Because features have very different scales (e.g., Income vs. binary flags), we wrap Logistic Regression in a **Pipeline** with **StandardScaler**. `class_weight='balanced'` is used to address the 90.4% / 9.6% class imbalance, consistent with the Decision Tree models.

In [ ]:
# Baseline Logistic Regression (balanced classes, default threshold = 0.5)
logreg1 = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42,
    )),
])
logreg1.fit(X_train, y_train)


In [ ]:
plot_confusion_matrix(logreg1, X_train, y_train, 'Logistic Regression (Baseline) — Training')
logreg1_train = model_performance_classification(logreg1, X_train, y_train)
print('Training performance:')
display(logreg1_train)

plot_confusion_matrix(logreg1, X_test, y_test, 'Logistic Regression (Baseline) — Test')
logreg1_test = model_performance_classification(logreg1, X_test, y_test)
print('Test performance:')
display(logreg1_test)


**Baseline Logistic Regression observations:**
- The model generalizes reasonably well — training and test metrics are closer than the default (unpruned) Decision Tree.
- With the default 0.5 threshold, Recall on the minority class may be lower than desired for campaign targeting.
- Threshold and regularization tuning can improve Recall without sacrificing too much Precision.

### Logistic Regression — Hyperparameter and Threshold Tuning

In [ ]:
# Hold out a validation split from training data for threshold tuning
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.20, stratify=y_train, random_state=42
)

# Tune regularization strength (C) using 5-fold CV with Recall as the objective
logreg_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        penalty='l2',
        class_weight='balanced',
        max_iter=1000,
        random_state=42,
    )),
])

param_grid = {'clf__C': [0.01, 0.1, 1, 10, 100]}
grid_search = GridSearchCV(
    logreg_pipe, param_grid, cv=5, scoring='recall', n_jobs=-1
)
grid_search.fit(X_tr, y_tr)

print('Best C from GridSearchCV:', grid_search.best_params_['clf__C'])
print('Best CV Recall:', round(grid_search.best_score_, 4))

# Refit best pipeline on full training set
logreg2 = grid_search.best_estimator_
logreg2.fit(X_train, y_train)

# Tune decision threshold on validation set to improve Recall
val_proba = logreg2.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, val_proba)

best_threshold = 0.5
target_recall = 0.85
for precision, recall, threshold in zip(precisions[:-1], recalls[:-1], thresholds):
    if recall >= target_recall:
        best_threshold = threshold
        break
else:
    best_threshold = thresholds[np.argmax(recalls[:-1])]

print(f'Selected decision threshold: {best_threshold:.4f}')


In [ ]:
plot_confusion_matrix(
    logreg2, X_train, y_train,
    'Logistic Regression (Tuned) — Training',
    threshold=best_threshold,
)
logreg2_train = model_performance_classification(
    logreg2, X_train, y_train, threshold=best_threshold
)
print('Training performance:')
display(logreg2_train)

plot_confusion_matrix(
    logreg2, X_test, y_test,
    'Logistic Regression (Tuned) — Test',
    threshold=best_threshold,
)
logreg2_test = model_performance_classification(
    logreg2, X_test, y_test, threshold=best_threshold
)
print('Test performance:')
display(logreg2_test)


**Tuned Logistic Regression observations:**
- **Ridge (L2) regularization** with cross-validated `C` reduces overfitting and handles correlated features (e.g., Age and Experience).
- **Threshold tuning** on a validation split shifts the trade-off toward higher Recall — fewer genuine loan prospects are missed.
- Coefficients remain interpretable: positive coefficients increase the log-odds of accepting a personal loan; negative coefficients decrease them.

In [ ]:
# Logistic Regression coefficient interpretation (tuned model)
feature_names = list(X_train.columns)
coefficients = logreg2.named_steps['clf'].coef_[0]
coef_df = (
    pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})
    .sort_values('Coefficient', key=abs, ascending=False)
    .reset_index(drop=True)
)
print(coef_df.to_string(index=False))

plt.figure(figsize=(9, 6))
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Standardized Coefficient (log-odds)')
plt.title('Logistic Regression Coefficients — Tuned Model', fontsize=13)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


**Coefficient observations:**
- **Income**, **CCAvg**, **CD_Account**, and **Education** typically show positive coefficients — higher values increase loan acceptance probability.
- **Age** and **Experience** often have overlapping signal; regularization helps stabilize their combined effect.
- Compared to the Decision Tree's hard splits, Logistic Regression captures a **global linear trend** across all customers.

## Model Performance Comparison and Final Model Selection

In [ ]:
# Compare all Decision Tree and Logistic Regression models
models_train = pd.concat(
    [dtree1_train.T, dtree2_train.T, dtree3_train.T, logreg1_train.T, logreg2_train.T],
    axis=1,
)
models_train.columns = [
    'Default DT',
    'Pre-Pruned DT',
    'Post-Pruned DT',
    'Logistic Regression (Baseline)',
    'Logistic Regression (Tuned)',
]
print('=== Training Performance Comparison ===')
display(models_train.round(4))

models_test = pd.concat(
    [dtree1_test.T, dtree2_test.T, dtree3_test.T, logreg1_test.T, logreg2_test.T],
    axis=1,
)
models_test.columns = [
    'Default DT',
    'Pre-Pruned DT',
    'Post-Pruned DT',
    'Logistic Regression (Baseline)',
    'Logistic Regression (Tuned)',
]
print('\n=== Test Performance Comparison ===')
display(models_test.round(4))

# Visual comparison on the test set (primary metric: Recall)
metrics_to_plot = ['Recall', 'F1']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, metrics_to_plot):
    models_test.loc[metric].sort_values(ascending=False).plot(
        kind='bar', ax=ax, color='steelblue', edgecolor='black'
    )
    ax.set_title(f'Test {metric} — All Models')
    ax.set_ylabel(metric)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.set_ylim(0, 1.05)
    for p in ax.patches:
        ax.annotate(
            f'{p.get_height():.3f}',
            (p.get_x() + p.get_width() / 2, p.get_height()),
            ha='center', va='bottom', fontsize=9,
        )
plt.tight_layout()
plt.show()

# Select final model: highest test Recall; break ties with F1
candidate_models = {
    'Default DT': dtree1,
    'Pre-Pruned DT': dtree2,
    'Post-Pruned DT': dtree3,
    'Logistic Regression (Baseline)': logreg1,
    'Logistic Regression (Tuned)': logreg2,
}
final_model_name = models_test.loc['Recall'].idxmax()
top_recall = models_test.loc['Recall'].max()
tied = models_test.loc['Recall'][models_test.loc['Recall'] == top_recall].index.tolist()
if len(tied) > 1:
    final_model_name = models_test.loc['F1', tied].idxmax()

final_model = candidate_models[final_model_name]
final_threshold = best_threshold if final_model_name == 'Logistic Regression (Tuned)' else None

print('\n=== Final Model Selection ===')
print(f'Selected model: {final_model_name}')
print(f'Test Recall: {models_test.loc["Recall", final_model_name]:.4f}')
print(f'Test F1:     {models_test.loc["F1", final_model_name]:.4f}')
if final_threshold is not None:
    print(f'Decision threshold: {final_threshold:.4f}')


**Model Comparison Analysis (Decision Tree vs Logistic Regression):**

| Model | Strengths | Limitations |
|-------|-----------|-------------|
| **Default DT** | Captures complex patterns | Severe overfitting; unreliable on unseen data |
| **Pre-Pruned DT** | Simple, interpretable rules; good generalization | May underfit; lower F1 than post-pruned tree |
| **Post-Pruned DT** | Strong Recall with interpretable decision rules | Step-wise splits; zero importance for unused features |
| **Logistic Regression (Baseline)** | Stable linear baseline; calibrated probabilities | Default 0.5 threshold may miss minority class |
| **Logistic Regression (Tuned)** | Higher Recall via threshold tuning; interpretable coefficients | Assumes linear log-odds; may miss sharp non-linear cutoffs |

**Selection criteria:**
1. **Primary metric — Test Recall:** identifies the most potential loan customers (minimizes false negatives).
2. **Secondary metric — Test F1:** ensures Precision does not collapse to unusable levels.
3. **Generalization:** training and test scores should be reasonably close.
4. **Business usability:** interpretability and actionable probability scores for campaign tiers.

**Final model:** The model with the **highest test Recall** is selected automatically in the code above (ties broken by F1). Review the comparison table and bar charts to confirm whether the **Post-Pruned Decision Tree** or **Tuned Logistic Regression** performs best on your run.

**Typical outcome:** The post-pruned Decision Tree often achieves competitive or higher Recall because it captures non-linear thresholds (e.g., Income cutoffs). Tuned Logistic Regression is frequently the better choice when **smooth probability scores** are needed for tiered marketing, even if Recall is slightly lower.


In [ ]:
# Feature importance / coefficients of the final model
feature_names = list(X_train.columns)

if final_model_name.startswith('Logistic Regression'):
    coefficients = final_model.named_steps['clf'].coef_[0]
    importances = np.abs(coefficients)
    xlabel = 'Absolute Standardized Coefficient'
    title = f'Feature Importance — Final Model ({final_model_name})'
    fi_df = (
        pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients, 'Importance': importances})
        .sort_values('Importance', ascending=False)
        .reset_index(drop=True)
    )
else:
    importances = final_model.feature_importances_
    xlabel = 'Relative Importance'
    title = f'Feature Importances — Final Model ({final_model_name})'
    fi_df = (
        pd.DataFrame({'Feature': feature_names, 'Importance': importances})
        .sort_values('Importance', ascending=False)
        .reset_index(drop=True)
    )

indices = np.argsort(importances)
plt.figure(figsize=(9, 6))
plt.title(title, fontsize=13)
plt.barh(range(len(indices)), importances[indices], color='steelblue', align='center')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel(xlabel)
plt.tight_layout()
plt.show()

print(fi_df.to_string(index=False))


**Final model interpretability:**
- If the **Decision Tree** is selected, review the tree visualization and decision rules above — splits on **Income**, **Family**, **Education**, and **CCAvg** are typically the most actionable for marketing.
- If **Logistic Regression** is selected, use the coefficient plot: features with the largest absolute coefficients drive acceptance probability the most. Positive coefficients increase loan uptake odds; negative coefficients decrease them.
- Both model families consistently highlight **Income** as a dominant driver, aligning with the EDA findings.


## Actionable Insights and Business Recommendations


**Summary of Key Findings:**
- Decision Tree models (especially post-pruned) and tuned Logistic Regression were compared on **Recall** (primary) and **F1** (secondary).
- The automatically selected final model minimizes missed loan prospects while maintaining reasonable precision.
- **Income** remains the strongest predictor across both model families; Logistic Regression adds coefficient-based insight while Decision Trees provide explicit if-then rules.

**Recommendations for the Marketing Team:**

1. **Prioritize high-income customers.** Income is the dominant predictor in both Decision Tree and Logistic Regression models. Campaigns should prioritize customers with annual incomes above ~$100K+.

2. **Consider family size in targeting.** Family size is a key split in Decision Trees and contributes meaningfully in Logistic Regression. Tailor offers for households with family size 3–4.

3. **Segment by Education level.** Advanced/Professional degree holders accept loans at higher rates. Messaging can appeal to financial growth and investment needs.

4. **Focus on high credit card spenders.** High CCAvg indicates active credit behavior — design offers aligned with spending patterns.

5. **Leverage CD Account holders.** EDA shows ~46% acceptance among CD holders vs ~7.5% for non-holders. Dedicated cross-sell offers remain valuable even when CD_Account has lower model importance (signal may overlap with Income).

6. **Use tiered campaign targeting based on model probability scores** (especially effective with Logistic Regression):
   - **High probability (>70%):** Direct outreach via relationship managers.
   - **Medium probability (40–70%):** Targeted digital/email campaigns.
   - **Low probability (<40%):** Low-cost awareness campaigns only.

7. **Choose the right model for the use case:**
   - Use **Decision Tree** rules for simple, explainable segmentation in field campaigns.
   - Use **Logistic Regression** probabilities for ranked lead lists and ROI-based budget allocation.

8. **Monitor and retrain.** Track campaign outcomes and periodically retrain both models with fresh data to account for evolving customer behavior.


___